In [1]:
# Completely remove pyspark
!pip uninstall -y pyspark dataproc-spark-connect -q

# Remove leftover spark folders (important)
!rm -rf /usr/local/lib/python3.12/dist-packages/pyspark
!rm -rf /usr/local/lib/python3.12/dist-packages/dataproc*

# Install Java 17
!apt-get update -qq
!apt-get install openjdk-17-jdk-headless -qq > /dev/null

# Set JAVA_HOME
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

# Install fresh pyspark 4.0.2
!pip install pyspark==4.0.2 -q

# Create Spark session
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("test") \
    .master("local[*]") \
    .getOrCreate()

spark

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
!java -version

openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


In [3]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("jonaskoenig/topic_classification")

df_pd = ds["train"].to_pandas()

print("Original rows:", len(df_pd))
df_pd.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Original rows: 13054978


,work,news,sports,music,movies,politics,phones,self-driving cars,family,cars,...,science,style,opinion,economy,history,technology,affair,development,mobility,text
0,0,0,0,0,0,0,0,0,1,0,...,1,0,0,0,0,0,0,0,1,"And this year, the number will be over 150,000..."
1,0,0,0,0,0,0,0,0,0,0,...,0,1,0,1,0,0,0,0,0,MR. SPICER: I think the campaign will make dec...
2,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,You dont have to test every person in the stat...
3,0,0,0,0,0,0,0,0,1,0,...,1,0,0,0,1,0,0,0,0,And Dr. Fauci is going to emphasize this about...
4,0,1,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,SANDERS: Certainly in a number of the conversa...


In [4]:
df_pd = df_pd.sample(frac=0.05, random_state=42)
print("Sample rows:", len(df_pd))

Sample rows: 652749


In [5]:
topic_columns = [
    'news','sports','music','movies','politics','phones',
    'family','cars','business','health','science',
    'technology','economy','history'
]

def get_label(row):
    for col in topic_columns:
        # Check if the column exists in the row's index before trying to access it
        if col in row.index and row[col] == 1:
            return col
    return "other"

df_pd["label"] = df_pd.apply(get_label, axis=1)
df_pd = df_pd[["text", "label"]]

In [8]:
df_pd.to_csv(f"{BASE_PATH}/data.csv", index=False)

In [9]:
df = spark.read.csv(f"{BASE_PATH}/data.csv", header=True, inferSchema=True)

# Repartition for distributed processing
df = df.repartition(200)

df.printSchema()

root
 |-- text: string (nullable = true)
 |-- label: string (nullable = true)



In [10]:
spark_df = spark.createDataFrame(df_pd)

spark_df.printSchema()
spark_df.show(5)

root
 |-- text: string (nullable = true)
 |-- label: string (nullable = true)

+--------------------+-------+
|                text|  label|
+--------------------+-------+
|This is how democ...|economy|
|I am also complet...|science|
|And they *hate* L...|  other|
|FYI, I haven't si...| sports|
|The more scare yo...| sports|
+--------------------+-------+
only showing top 5 rows


In [16]:
import os

output_dir = '/content/project/data/schemas/'
os.makedirs(output_dir, exist_ok=True)

with open(os.path.join(output_dir, "schema.txt"), "w") as f:
    f.write(str(spark_df.schema))

In [13]:
spark_df.write.mode("overwrite").parquet("/content/project/data/samples/topic_parquet")

# **2_feature_engineering**

In [14]:
df = spark.read.parquet("/content/project/data/samples/topic_parquet")
df.show(5)

+--------------------+-------+
|                text|  label|
+--------------------+-------+
|This is how democ...|economy|
|I am also complet...|science|
|And they *hate* L...|  other|
|FYI, I haven't si...| sports|
|The more scare yo...| sports|
+--------------------+-------+
only showing top 5 rows


In [17]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF, StringIndexer
from pyspark.ml import Pipeline

# Fill null values in the 'text' column with an empty string
df = df.fillna('', subset=['text'])

tokenizer = Tokenizer(inputCol="text", outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
hashingTF = HashingTF(inputCol="filtered_words", outputCol="tf", numFeatures=10000)
idf = IDF(inputCol="tf", outputCol="features")
label_indexer = StringIndexer(inputCol="label", outputCol="label_index")

feature_pipeline = Pipeline(stages=[
    tokenizer,
    remover,
    hashingTF,
    idf,
    label_indexer
])

feature_model = feature_pipeline.fit(df)
df_features = feature_model.transform(df)

df_features.select("features", "label_index").show(5)

+--------------------+-----------+
|            features|label_index|
+--------------------+-----------+
|(10000,[157,1901,...|        9.0|
|(10000,[3792,6240...|        1.0|
|(10000,[2441,3910...|        0.0|
|(10000,[1152,2114...|        2.0|
|(10000,[2483,6103...|        2.0|
+--------------------+-----------+
only showing top 5 rows


In [18]:
df_features.write.mode("overwrite").parquet("/content/project/data/samples/features_parquet")

# **model_training**

In [19]:
df = spark.read.parquet("/content/project/data/samples/features_parquet")

In [20]:
train, temp = df.randomSplit([0.7, 0.3], seed=42)
val, test = temp.randomSplit([0.5, 0.5], seed=42)

print(train.count(), val.count(), test.count())

456870 98203 97676


**Models**

In [30]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.classification import NaiveBayes
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.classification import RandomForestClassifier

## **Logistic Regression Model**

In [24]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label_index",
    maxIter=10
)

model_lr = lr.fit(train)

pred_lr_val = model_lr.transform(val)
pred_lr_val.select("label_index", "prediction").show(5)

+-----------+----------+
|label_index|prediction|
+-----------+----------+
|        3.0|       0.0|
|        4.0|       1.0|
|        3.0|       3.0|
|        0.0|       1.0|
|        7.0|       2.0|
+-----------+----------+
only showing top 5 rows


## **Naive Bayes Model**

In [27]:
nb = NaiveBayes(
    featuresCol="features",
    labelCol="label_index"
)

model_nb = nb.fit(train)

pred_nb_val = model_nb.transform(val)
pred_nb_val.select("label_index", "prediction").show(5)

+-----------+----------+
|label_index|prediction|
+-----------+----------+
|        3.0|       0.0|
|        4.0|       3.0|
|        3.0|       3.0|
|        0.0|       0.0|
|        7.0|       7.0|
+-----------+----------+
only showing top 5 rows


## **Decision Tree**

In [32]:
dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label_index",
    maxDepth=10
)

model_dt = dt.fit(train)

pred_dt_val = model_dt.transform(val)
pred_dt_val.select("label_index", "prediction").show(5)

+-----------+----------+
|label_index|prediction|
+-----------+----------+
|        3.0|       0.0|
|        4.0|       1.0|
|        3.0|       1.0|
|        0.0|       1.0|
|        7.0|       1.0|
+-----------+----------+
only showing top 5 rows


## **Random Forest Model**

In [33]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label_index",
    numTrees=20
)

model_rf = rf.fit(train)

pred_rf_val = model_rf.transform(val)
pred_rf_val.select("label_index", "prediction").show(5)

+-----------+----------+
|label_index|prediction|
+-----------+----------+
|        3.0|       0.0|
|        4.0|       0.0|
|        3.0|       0.0|
|        0.0|       0.0|
|        7.0|       0.0|
+-----------+----------+
only showing top 5 rows


In [34]:
import os

model_base_path = "/content/project/models"

os.makedirs(model_base_path, exist_ok=True)

In [35]:
model_lr.save("/content/project/models/lr_model")
model_nb.save("/content/project/models//nb_model")
model_dt.save("/content/project/models/dt_model")
model_rf.save("/content/project/models/rf_model")

print("All models saved successfully.")

All models saved successfully.


# **Evalution**

In [36]:
import os
import pandas as pd

from pyspark.ml.classification import LogisticRegressionModel
from pyspark.ml.classification import NaiveBayesModel
from pyspark.ml.classification import DecisionTreeClassificationModel
from pyspark.ml.classification import RandomForestClassificationModel

from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [37]:
data_path = "/content/project/data/samples/features_parquet"

df = spark.read.parquet(data_path)

print("Total records:", df.count())
df.printSchema()

Total records: 652749
root
 |-- text: string (nullable = true)
 |-- label: string (nullable = true)
 |-- words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- filtered_words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- tf: vector (nullable = true)
 |-- features: vector (nullable = true)
 |-- label_index: double (nullable = true)



In [38]:
train_df, temp_df = df.randomSplit([0.7, 0.3], seed=42)
validation_df, test_df = temp_df.randomSplit([0.5, 0.5], seed=42)

print("Training records:", train_df.count())
print("Validation records:", validation_df.count())
print("Test records:", test_df.count())

Training records: 456870
Validation records: 98203
Test records: 97676


In [39]:
model_base_path = "/content/project/models"

model_lr = LogisticRegressionModel.load(model_base_path + "/lr_model")
model_nb = NaiveBayesModel.load(model_base_path + "/nb_model")
model_dt = DecisionTreeClassificationModel.load(model_base_path + "/dt_model")
model_rf = RandomForestClassificationModel.load(model_base_path + "/rf_model")

print("All models loaded successfully.")

All models loaded successfully.


In [40]:
evaluator = MulticlassClassificationEvaluator(
    labelCol="label_index",
    predictionCol="prediction",
    metricName="accuracy"
)

In [41]:
pred_lr_test = model_lr.transform(test_df)
accuracy_lr = evaluator.evaluate(pred_lr_test)

print("Logistic Regression Accuracy:", accuracy_lr)

Logistic Regression Accuracy: 0.4388488472091404


In [42]:
pred_nb_test = model_nb.transform(test_df)
accuracy_nb = evaluator.evaluate(pred_nb_test)

print("Naive Bayes Accuracy:", accuracy_nb)

Naive Bayes Accuracy: 0.2924362176993325


In [43]:
pred_dt_test = model_dt.transform(test_df)
accuracy_dt = evaluator.evaluate(pred_dt_test)

print("Decision Tree Accuracy:", accuracy_dt)

Decision Tree Accuracy: 0.38500757606781605


In [44]:
pred_rf_test = model_rf.transform(test_df)
accuracy_rf = evaluator.evaluate(pred_rf_test)

print("Random Forest Accuracy:", accuracy_rf)

Random Forest Accuracy: 0.31231827675170976


In [45]:
results = [
    ("Logistic Regression", accuracy_lr),
    ("Naive Bayes", accuracy_nb),
    ("Decision Tree", accuracy_dt),
    ("Random Forest", accuracy_rf)
]

results_df = pd.DataFrame(results, columns=["Model", "Accuracy"])

print("Model Comparison Results:")
print(results_df)

Model Comparison Results:
                 Model  Accuracy
0  Logistic Regression  0.438849
1          Naive Bayes  0.292436
2        Decision Tree  0.385008
3        Random Forest  0.312318


In [46]:
output_path = "/content/project/outputs"
os.makedirs(output_path, exist_ok=True)

results_df.to_csv(output_path + "/model_results.csv", index=False)

print("Results saved successfully at:", output_path)

Results saved successfully at: /content/project/outputs
